# Lesson 07 - အစီအစဉ်ဖော်ဆောင်ရေး ဒီဇိုင်းပုံစံ

ဤ notebook သည် Microsoft Agent Framework ကို အသုံးပြု၍ AI ကိုယ်စားလှယ်များအတွက် **အစီအစဉ်ဖော်ဆောင်ရေး ဒီဇိုင်းပုံစံ** ကို ဖော်ပြသည်။
သင်သည် ခက်ခဲသော ခရီးသွားတောင်းဆိုမှုများကို ဖွဲ့စည်းထားသော အပိုင်းဆိုင်ရာ အလုပ်များသို့ ခွဲထုတ်ခြင်း၊ 
အထူးပြု ကိုယ်စားလှယ်များသို့ ခန့်အပ်ခြင်းနှင့် Pydantic မော်ဒယ်များဖြင့် စွမ်းဆောင်သူ ဖွဲ့စည်းထားသော output ကို အသုံးပြုကာ ရလဒ်အစီအစဉ်ကို အကောင်အထည် ဖော်ဆောင်နည်းများကို သင်ယူသွားမည်။


## ဆက်တင်ပြင်ဆင်မှု


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

Task decomposition သည် စီမံချက်ဒီဇိုင်းပုံစံ၏ အဓိကအချက်ဖြစ်သည်။ တစ်ဦးတည်းသော အေးဂျင့်ကို ကုသိုလ်စုံလင်သော တောင်းဆိုချက်ကို အဆုံးအထိ ကိုင်တွယ်စေခြင်းမပြုဘဲ ပြဿနာကို ပိုမိုသေးငယ်ပြီး၊ ပြတ်သားစွာ သတ်မှတ်ထားသည့် **subtasks** များသို့ ခွဲခြားသည်။ subtasks တစ်ခုစီကို အထူးပြု အေးဂျင့်တစ်ဦး (ဥပမာ- လေကြောင်း၊ ဟိုတယ်များ၊ လှုပ်ရှားမှုများ) ထံသို့ ဦးစားပေးမှုနဲ့ တာဝန်ဆိုင်ရာ အဆင့်သတ်မှတ်ချက်များဖြင့် ပေးအပ်သည်။

ဤနည်းလမ်းသည် အောက်ပါအကျိုးကျေးဇူးများကို ပေးစွမ်းသည်-
- **ရှင်းလင်းမှု**: subtasks တစ်ခုစီတွင် တာဝန်တစ်ခုသာရှိသည်။
- **병렬စွမ်းဆောင်မှု**: လွတ်လပ်သော subtasks များကို တပြိုင်နက်တွင် လည်ပတ်နိုင်သည်။
- **ယုံကြည်နိုင်မှု**: မအောင်မြင်မှုများမှာ subtasks တစ်ခုချင်းစီတွင်သာ ခွဲခြားသိမ်းဆည်းထားသည်။
- **ဘတ်ဂျက်တွင် ကြည့်ရှုသိမ်းဆည်းမှု**: စုဆောင်းချက်များကို တစ်ခုချင်း subtask ပေါ်တွင် ခန့်မှန်းပြီး စုစည်းသည်။


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## အစီအစဉ်ဖြစ်စေသော Agent တစ်ယောက်ကို ဖန်တီးခြင်းနှင့် ဖွဲ့စည်းထားသော အထွက်ရှိခြင်း

အစီအစဉ် agent သည် **ရှေ့စာရင်းညွှန်ကြားသူ** အဖြစ် အလုပ်လုပ်သည်။ အဆင့်မြင့် ခရီးသွား တောင်းဆိုချက်တစ်ခုရရှိသည့်အခါ
၎င်းသည် ဖွဲ့စည်းထားသော `TravelPlan` တစ်ခုကို ထုတ်ပေးသည် — တောင်းဆိုချက်ကို အုပ်စုခွဲ၍၊ ဦးစားပေးရာ စနစ်များထား၍၊
နှင့် လိုအပ်ချက်များ သတ်မှတ်၍ concierge သို့မဟုတ် အကောင်အထည်ဖော်ခြင်းအပြင်အဆင်အဖွဲ့တစ်ခုက အလုပ်ကို ဆောင်ရွက်နိုင်စေရန်။


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## အထူးပြုကိရိယာများဖြင့် စီမံကိန်းတစ်ခုကို ဆောင်ရွက်ခြင်း

ရှေ့ထပ်က အေးဂျင့်သည် ဖွဲ့စည်းထားသော စီမံကိန်းကို ထုတ်လုပ်ပြီးပါက **concierge agent** သည် ၎င်းကို ဆောင်ရွက်သည်။
တိုင်းတာရမည့် အထူးပြုကိရိယာတစ်ခုစီသည် အခြားအုပ်စုတစ်ခုစီကို ကိုင်တွယ်သည် (လေကြောင်းလိုင်းများ၊ ဟိုတယ်များ၊ လှုပ်ရှားမှုများ)။
Concierge သည် စီမံကိန်း၏ အွန်လိုင်းလုပ်ငန်းများကို မူလီခွဲအမီ မျိုးစုံထားပြီး သင့်တော်သော ကိရိယာသို့ တစ်ခုချင်းစီကို ပို့ပေးသည်။


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ သင်သည် AI အေးဂျင့်များအတွက် **အစီအစဉ်ရေးဆွဲမှု ဒီဇိုင်းပုံစံ** ကိုလေ့လာခဲ့သည်-

1. **အလုပ်ခွဲခြားခြင်း** — ရုံးရှေ့တန်းအစီအစဉ်ရေးဆွဲအေးဂျင့်တစ်ဦးသည် ခက်ခဲသောခရီးသွားမေတ္တာတောင်းဆိုမှုကို Pydantic မော်ဒယ်များကို အသုံးပြုပြီး ဖွဲ့စည်းထားသောအလုပ်ခွဲများသို့ခွဲခြားပြီး၊ ထူးချွန်နေရာအေးဂျင့်တစ်ဦးချင်းစီအား ဦးစားပေးမှုများနှင့် ဆက်စပ်မှုများစွာဖြင့်ပေးအပ်သည်။
2. **ဖွဲ့စည်းတည်ဆောက်ထားသောအထွက်** — `response_format` ကို ပေးပို့ခြင်းဖြင့် အေးဂျင့်သည် အပြားစုံစစ်ပြီး `TravelPlan` ကို ပုံမှန်စာသားမဟုတ်ဘဲ ပြန်လည်ပေးပို့ပြီး၊ နောက်တစ်ဆင့် ဆောင်ရွက်မှုများကို ယုံကြည်စိတ်ချရအောင်ပြုလုပ်သည်။
3. **အစီအစဉ်အကောင်အထည်ဖော်ခြင်း** — ကွန်ဆီရျ့ခ်အေးဂျင့်သည် အထူးကိရိယာများ (`book_flight`, `reserve_hotel`, `book_activity`) ကို အသုံးပြုပြီး အလုပ်ခွဲများအား တစ်ဆင့်ချင်းစီ ဆက်လက်အကောင်အထည်ဖော်ကာ ရလဒ်များကို အစီရင်ခံသည်။

ဒီပုံစံသည် *ဘာလုပ်မည်နည်း* (အစီအစဉ်ရေးဆွဲခြင်း) ကို *ဘယ်လိုလုပ်မည်နည်း* (အကောင်အထည်ဖော်ခြင်း) မှ ခွဲထုတ်ထားပြီး၊ အေးဂျင့်များကို လွယ်ကူစွာ module ပြုလုပ်နိုင်၊ စမ်းသပ်နိုင်၊ နှင့် တိုးချဲ့ရန်လွယ်ကူစေသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အတည်မပြုခြင်း**  
ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သည့် [Co-op Translator](https://github.com/Azure/co-op-translator) ကို အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် ကြိုးပမ်းပါသောကြောင့်ဖြစ်သော်လည်း၊ အလိုအလျောက်ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မမှန်ကန်သော အချက်များ ပါဝင်နိုင်မှုရှိကြောင်း သတိပြုရန်လိုအပ်ပါသည်။ မူလ စာရွက်စာတမ်းကို သဘာဝဘာသာဖြင့် ဆက်လက်ကိုင်တွယ်ရမည့် တရားဝင်အရင်းအမြစ်အဖြစ် အသိအမှတ်ပြုရပါမည်။ အရေးပါတဲ့ အချက်အလက်များအတွက်တော့ လူ့ အမှုဆောင်ဘာသာပြန်မှူးတစ်ဦးကို အဆိုပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုမှုမှ ဆင်းရဲနိမိတ်ခြင်းများ သို့မဟုတ် မှားယွင်းစွာ နားလည်ခြင်းများအတွက် ကျွန်ုပ်တို့မှာ တာဝန်မရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
